# Import Library

In [ ]:
!pip install Sastrawi
!pip install imblearn
!pip install tqdm

In [ ]:
import pandas as pd
pd.options.mode.chained_assignment = None
from tqdm import tqdm
tqdm.pandas()

import numpy as np
import datetime as dt
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Cleaned Dataset

In [ ]:
data = pd.read_csv('ulasan_game_roblox_cleaned.csv')
data

,Unnamed: 0,reviewId,userName,userImage,content,score,thumbsUpCount,at
0,0,ff479260-2a43-4550-9862-3a0d16879170,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jelek ngelag,1,0,2026-05-09 19:03:41
1,1,11eef1d9-bcd9-479c-ad83-69b493b3cf55,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,woi Roblox tolong bantuin akun gw kembalikan s...,5,0,2026-05-09 19:02:00
2,2,deb90458-cc3b-4cd2-aeca-2b517f62fd6c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ini gimana akun saya tidak bisa di masuki udah...,1,1,2026-05-09 19:01:53
3,3,5ab68163-0c9c-4d77-a04e-486df6c22aef,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus banget,5,0,2026-05-09 19:01:06
4,4,a7695c25-7ec2-401e-9a75-46598d6df695,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,apaan sih gua cuma putar lagu langsung di gak ...,2,0,2026-05-09 19:00:35
...,...,...,...,...,...,...,...,...
9995,9995,a127c1c4-281c-4b41-bfde-02ce120cdb16,Asvi Maulidya,https://play-lh.googleusercontent.com/a-/ALV-U...,"game nya seru sih,tapi tolong dong jangan veri...",3,0,2026-04-30 14:38:34
9996,9996,24b84cc6-70f2-48dd-b464-13d22cf02e0b,KPRTANPARIBET,https://play-lh.googleusercontent.com/a/ACg8oc...,seru banget mainnya,5,0,2026-04-30 14:38:15
9997,9997,62a78373-fea8-45c0-b655-542c5f13dc8f,Alfarizki Rendra,https://play-lh.googleusercontent.com/a/ACg8oc...,game ini seru banyak map nya tapi sayang udah ...,4,0,2026-04-30 14:34:25
9998,9998,fde16f07-7c77-4731-875b-27567f46f0a7,Fitri Handayani,https://play-lh.googleusercontent.com/a/ACg8oc...,kembalikan roblox yang dulu!!! 😭,1,0,2026-04-30 14:33:21


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Unnamed: 0     10000 non-null  int64 
 1   reviewId       10000 non-null  object
 2   userName       10000 non-null  object
 3   userImage      10000 non-null  object
 4   content        9998 non-null   object
 5   score          10000 non-null  int64 
 6   thumbsUpCount  10000 non-null  int64 
 7   at             10000 non-null  object
dtypes: int64(3), object(5)
memory usage: 625.1+ KB


In [ ]:
# Fill null value with empty string
data['content'] = data['content'].fillna('gak ada')

# Labelling

In [ ]:
# Inisialisasi daftar sentimen
label_num = []

# Iterasi setiap baris dalam DataFrame
for index, row in data.iterrows():
    if row['score'] > 3 :
        label_num.append(1)     # nilai 1 untuk score 4 - 5
    elif row['score'] == 3:
        label_num.append(0)     # nilai 0 untuk score 3
    else:
        label_num.append(-1)    # nilai -1 untuk score 1 - 2

# Tambahkan kolom baru
data['label_num'] = label_num
data.head()

,Unnamed: 0,reviewId,userName,userImage,content,score,thumbsUpCount,at,label_num
0,0,ff479260-2a43-4550-9862-3a0d16879170,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jelek ngelag,1,0,2026-05-09 19:03:41,-1
1,1,11eef1d9-bcd9-479c-ad83-69b493b3cf55,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,woi Roblox tolong bantuin akun gw kembalikan s...,5,0,2026-05-09 19:02:00,1
2,2,deb90458-cc3b-4cd2-aeca-2b517f62fd6c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ini gimana akun saya tidak bisa di masuki udah...,1,1,2026-05-09 19:01:53,-1
3,3,5ab68163-0c9c-4d77-a04e-486df6c22aef,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus banget,5,0,2026-05-09 19:01:06,1
4,4,a7695c25-7ec2-401e-9a75-46598d6df695,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,apaan sih gua cuma putar lagu langsung di gak ...,2,0,2026-05-09 19:00:35,-1


In [ ]:
# Polarity Label
label = []

# Iterasi setiap baris dalam DataFrame
for index, row in data.iterrows():
    if row['score'] > 3 :
        label.append("positive")     # nilai 1 untuk score 4 - 5
    elif row['score'] == 3:
        label.append("neutral")     # nilai 0 untuk score 3
    else:
        label.append("negative")    # nilai -1 untuk score 1 - 2

# Tambahkan kolom baru
data['polarity'] = label
data.head()

,Unnamed: 0,reviewId,userName,userImage,content,score,thumbsUpCount,at,label_num,polarity
0,0,ff479260-2a43-4550-9862-3a0d16879170,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jelek ngelag,1,0,2026-05-09 19:03:41,-1,negative
1,1,11eef1d9-bcd9-479c-ad83-69b493b3cf55,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,woi Roblox tolong bantuin akun gw kembalikan s...,5,0,2026-05-09 19:02:00,1,positive
2,2,deb90458-cc3b-4cd2-aeca-2b517f62fd6c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ini gimana akun saya tidak bisa di masuki udah...,1,1,2026-05-09 19:01:53,-1,negative
3,3,5ab68163-0c9c-4d77-a04e-486df6c22aef,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus banget,5,0,2026-05-09 19:01:06,1,positive
4,4,a7695c25-7ec2-401e-9a75-46598d6df695,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,apaan sih gua cuma putar lagu langsung di gak ...,2,0,2026-05-09 19:00:35,-1,negative


# Preprocessing Text

In [ ]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [ ]:
# Membaca kamus slang dari file Excel
slang_words = pd.read_excel('kamuskatabaku.xlsx')

# Menyusun kamus slang sebagai dictionary: slang -> kata baku
slang_dict = dict(zip(slang_words['tidak_baku'], slang_words['kata_baku']))

# Fungsi untuk mengganti slang dengan kata baku
def fix_slangwords(text):
    return ' '.join([slang_dict.get(words, words) for words in text.split()])

In [ ]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
data['text_clean'] = data['content'].progress_apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
data['text_casefoldingText'] = data['text_clean'].progress_apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
data['text_slangwords'] = data['text_casefoldingText'].progress_apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
data['text_tokenizingText'] = data['text_slangwords'].progress_apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
data['text_stopword'] = data['text_tokenizingText'].progress_apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
data['text_akhir'] = data['text_stopword'].progress_apply(toSentence)

100%|██████████| 10000/10000 [00:00<00:00, 81908.16it/s]


In [ ]:
data.head()

,Unnamed: 0,reviewId,userName,userImage,content,score,thumbsUpCount,at,label_num,polarity,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir
0,0,ff479260-2a43-4550-9862-3a0d16879170,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jelek ngelag,1,0,2026-05-09 19:03:41,-1,negative,jelek ngelag,jelek ngelag,jelek ngelag,"[jelek, ngelag]","[jelek, ngelag]",jelek ngelag
1,1,11eef1d9-bcd9-479c-ad83-69b493b3cf55,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,woi Roblox tolong bantuin akun gw kembalikan s...,5,0,2026-05-09 19:02:00,1,positive,woi Roblox tolong bantuin akun gw kembalikan s...,woi roblox tolong bantuin akun gw kembalikan s...,woi roblox tolong bantuin akun gue kembalikan ...,"[woi, roblox, tolong, bantuin, akun, gue, kemb...","[roblox, tolong, bantuin, akun, gue, kembalika...",roblox tolong bantuin akun gue kembalikan curi...
2,2,deb90458-cc3b-4cd2-aeca-2b517f62fd6c,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ini gimana akun saya tidak bisa di masuki udah...,1,1,2026-05-09 19:01:53,-1,negative,ini gimana akun saya tidak bisa di masuki udah...,ini gimana akun saya tidak bisa di masuki udah...,ini bagaimana akun saya tidak bisa di masuki s...,"[ini, bagaimana, akun, saya, tidak, bisa, di, ...","[akun, masuki, uninstal, sandi, pakai, no, way...",akun masuki uninstal sandi pakai no way indone...
3,3,5ab68163-0c9c-4d77-a04e-486df6c22aef,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus banget,5,0,2026-05-09 19:01:06,1,positive,bagus banget,bagus banget,bagus banget,"[bagus, banget]","[bagus, banget]",bagus banget
4,4,a7695c25-7ec2-401e-9a75-46598d6df695,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,apaan sih gua cuma putar lagu langsung di gak ...,2,0,2026-05-09 19:00:35,-1,negative,apaan sih gua cuma putar lagu langsung di gak ...,apaan sih gua cuma putar lagu langsung di gak ...,apaan sih gua cuma putar lagu langsung di tida...,"[apaan, sih, gua, cuma, putar, lagu, langsung,...","[gua, putar, lagu, langsung, onn, mic, menit, ...",gua putar lagu langsung onn mic menit tolong p...


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Unnamed: 0            10000 non-null  int64 
 1   reviewId              10000 non-null  object
 2   userName              10000 non-null  object
 3   userImage             10000 non-null  object
 4   content               10000 non-null  object
 5   score                 10000 non-null  int64 
 6   thumbsUpCount         10000 non-null  int64 
 7   at                    10000 non-null  object
 8   label_num             10000 non-null  int64 
 9   polarity              10000 non-null  object
 10  text_clean            10000 non-null  object
 11  text_casefoldingText  10000 non-null  object
 12  text_slangwords       10000 non-null  object
 13  text_tokenizingText   10000 non-null  object
 14  text_stopword         10000 non-null  object
 15  text_akhir            10000 non-null 

In [ ]:
data.isnull().sum()

,0
Unnamed: 0,0
reviewId,0
userName,0
userImage,0
content,0
score,0
thumbsUpCount,0
at,0
label_num,0
polarity,0


In [ ]:
# Simpan data hasil preprocessing
data.to_csv('ulasan_game_roblox_preprocessed_text.csv')

# Build Model

## Import Library

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report , confusion_matrix , accuracy_score

## Data Splitting

In [ ]:
X = data['text_akhir']
y = data['label_num']

## Extraction Feature Text

In [ ]:
# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X)

## Handling Imbalanced

In [ ]:
smote = SMOTE()
X_sm, y_sm = smote.fit_resample(X_tfidf, y)

## Build a Machine Learning Model

### Train Test Split Dataset (Training Set = 70% and Test Set = 30%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_sm, y_sm, test_size=0.3, random_state=42)

### Random Forest Classifier

In [ ]:
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train, y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train)
y_pred_test_rf = random_forest.predict(X_test)

# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9784921892687344
Random Forest - accuracy_test: 0.8563127311146329


### SVC

In [ ]:
# Membuat objek model SVC
svc = SVC()

# Melatih model SVC pada data pelatihan
svc.fit(X_train, y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_svc = svc.predict(X_train)
y_pred_test_svc = svc.predict(X_test)

# Evaluasi akurasi model SVC
accuracy_train_svc = accuracy_score(y_pred_train_svc, y_train)
accuracy_test_svc = accuracy_score(y_pred_test_svc, y_test)

# Menampilkan akurasi
print('SVC - accuracy_train:', accuracy_train_svc)
print('SVC - accuracy_test:', accuracy_test_svc)

SVC - accuracy_train: 0.9135159610595427
SVC - accuracy_test: 0.840112695897165


### Decision Tree Classifier

In [ ]:
# Membuat objek model DecisionTree
dt = DecisionTreeClassifier()

# Melatih model DecisionTree pada data pelatihan
dt.fit(X_train, y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = dt.predict(X_train)
y_pred_test_dt = dt.predict(X_test)

# Evaluasi akurasi model
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)

# Menampilkan akurasi
print('DecisionTree - accuracy_train:', accuracy_train_dt)
print('DecisionTree - accuracy_test:', accuracy_test_dt)

DecisionTree - accuracy_train: 0.9784921892687344
DecisionTree - accuracy_test: 0.7823560486001057


### Logistic Regression

In [ ]:
logistic_regression = LogisticRegression()

# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train, y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train)
y_pred_test_lr = logistic_regression.predict(X_test)

# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)

# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)

# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.8166176137649989
Logistic Regression - accuracy_test: 0.7481951047719669


### Train Test Split Dataset (Training Set = 75% and Test Set = 25%)

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_sm, y_sm, test_size=0.25, random_state=42)

### Random Forest Classifier

In [ ]:
# Membuat objek model Random Forest
random_forest_2 = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest_2.fit(X_train_2, y_train_2)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf_2 = random_forest.predict(X_train_2)
y_pred_test_rf_2 = random_forest.predict(X_test_2)

# Evaluasi akurasi model Random Forest
accuracy_train_rf_2 = accuracy_score(y_pred_train_rf_2, y_train_2)
accuracy_test_rf_2 = accuracy_score(y_pred_test_rf_2, y_test_2)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf_2)
print('Random Forest - accuracy_test:', accuracy_test_rf_2)

Random Forest - accuracy_train: 0.9701345354652391
Random Forest - accuracy_test: 0.8569617578702725


# WordCloud